In [18]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from src import content_based

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from src.split import temporal_split
from src.hybrid import hybrid_scores, top_n, normalize
from src import evaluation as ev

ratings = pd.read_csv(ROOT / "data" / "raw" / "ratings.csv")
movies = pd.read_csv(ROOT / "data" / "raw" / "movies.csv")
train, val, test = temporal_split(ratings)
print(train.shape, val.shape, test.shape)

(80896, 4) (10122, 4) (9818, 4)


In [19]:
K = 10
LIKE = 4.0

relevant_by_user = {
    u: set(g.loc[g["rating"] >= LIKE, "movieId"]) for u, g in val.groupby("userId")
}
relevant_by_user = {u: s for u, s in relevant_by_user.items() if s}
seen_by_user = train.groupby("userId")["movieId"].apply(set).to_dict()
all_movies = sorted(movies["movieId"].unique())

print("Utilisateurs évalués :", len(relevant_by_user))

Utilisateurs évalués : 565


In [20]:
C = train["rating"].mean()
m = 50
stats = train.groupby("movieId")["rating"].agg(["count", "mean"])
stats["score"] = (stats["count"] / (stats["count"] + m)) * stats["mean"] \
               + (m / (stats["count"] + m)) * C
pop_scores = stats["score"].to_dict()


def popularity_scores(user_id, movie_ids):
    return {mid: pop_scores.get(mid, C) for mid in movie_ids}


def random_scores(user_id, movie_ids):
    rng = np.random.default_rng(user_id)
    return dict(zip(movie_ids, rng.random(len(movie_ids))))

In [21]:
import time
t = time.time()
results, recs = evaluate(popularity_scores, content_based.get_scores, alphas)
print("Durée :", round(time.time() - t, 1), "s")
results

Durée : 107.0 s


,alpha,P@10,R@10,NDCG@10
0,0.0,0.003009,0.009323,0.005826
1,0.1,0.005664,0.011168,0.010205
2,0.2,0.009204,0.018277,0.016621
3,0.3,0.014159,0.024755,0.024462
4,0.4,0.018761,0.029771,0.031280
5,0.5,0.026195,0.041205,0.041068
6,0.6,0.032212,0.051392,0.050605
7,0.7,0.034867,0.051773,0.055518
8,0.8,0.036637,0.055093,0.058206
9,0.9,0.035221,0.053460,0.056601


## juste une petite interpretation

si sik alpha = 0 --> les genre apportnt de l'information
si = 1 normal les genres seules sont un signal faible 
le maximun de alpha intermidiaire --> la resulta que l hybride cherche 

In [22]:
def evaluate(collab_fn, content_fn, alphas):
    recs = {a: {} for a in alphas}
    for u in relevant_by_user:
        candidates = [mid for mid in all_movies if mid not in seen_by_user.get(u, set())]
        collab = normalize(collab_fn(u, candidates))
        content = normalize(content_fn(u, candidates))
        for a in alphas:
            scores = hybrid_scores(collab, content, a)
            recs[a][u] = [mid for mid, _ in top_n(scores, K)]

    rows = []
    for a in alphas:
        rows.append({
            "alpha": a,
            "P@10": ev.mean_at_k(ev.precision_at_k, recs[a], relevant_by_user, K),
            "R@10": ev.mean_at_k(ev.recall_at_k, recs[a], relevant_by_user, K),
            "NDCG@10": ev.mean_at_k(ev.ndcg_at_k, recs[a], relevant_by_user, K),
        })
    return pd.DataFrame(rows), recs


alphas = [round(a, 1) for a in np.arange(0, 1.01, 0.1)]
results, recs = evaluate(popularity_scores, random_scores, alphas)
results

,alpha,P@10,R@10,NDCG@10
0,0.0,0.000885,0.000768,0.000725
1,0.1,0.007788,0.012794,0.014437
2,0.2,0.012389,0.020719,0.020520
3,0.3,0.014867,0.024397,0.024086
4,0.4,0.017168,0.027747,0.026957
5,0.5,0.020354,0.033808,0.032055
6,0.6,0.023009,0.038592,0.035833
7,0.7,0.027080,0.043360,0.042185
8,0.8,0.029204,0.046903,0.046919
9,0.9,0.032035,0.047579,0.049996


In [23]:
best = results.loc[results["NDCG@10"].idxmax()]
print("Meilleur alpha (Validation, NDCG@10) :", best["alpha"])

best_recs = recs[best["alpha"]]
coverage = len({mid for l in best_recs.values() for mid in l}) / len(all_movies)
print(f"Couverture du catalogue : {coverage:.1%}")

Meilleur alpha (Validation, NDCG@10) : 1.0
Couverture du catalogue : 0.7%


**Interprétation (réglage de alpha sur Validation, 565 utilisateurs)**

- Content-Based seul (alpha = 0) : NDCG@10 = 0,0058, soit environ 8 fois l'aléatoire (0,0007) mais bien en dessous de la popularité.
- Popularité seule (alpha = 1) : P@10 = 0,0343, R@10 = 0,0551, NDCG@10 = 0,0541 (baseline à battre).
- Meilleur mélange : alpha = 0,8, avec P@10 = 0,0366 et NDCG@10 = 0,0582 (+7,5 % en NDCG par rapport à la popularité). Le rappel est identique.
- Le gain est modeste et mesuré sur un seul split : à confirmer avec le SVD à la place de la popularité.
- Couverture du catalogue : [à compléter avec les résultats].
- alpha est choisi sur Validation. Test ne sera utilisé qu'une seule fois, à la fin.